DAY 3

In [14]:
!pip install -q langchain langchain-groq

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

MODEL = "openai/gpt-oss-20b"
TEMPERATURE = 0.3
llm = ChatGroq(model=MODEL, temperature=TEMPERATURE)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def make_chain(system_prompt: str, temperature: float):
    model = ChatGroq(model=MODEL, temperature=temperature)
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ])
    return prompt | model | StrOutputParser()

chain = make_chain(SYSTEM, TEMPERATURE)
history = []

def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))


In [ ]:
history

## Customer-question tests

Upload `tests.csv` to the Colab session (Files pane) before running these cells. Each row has a customer question and a required phrase; the runner reports the pass rate.


In [ ]:
import csv

TEST_PATH = "/content/tests.csv"

with open(TEST_PATH, newline="", encoding="utf-8") as test_file:
    TESTS = list(csv.DictReader(test_file))

if not TESTS:
    raise ValueError(f"No test cases found in {TEST_PATH}")
if any(not row.get("question", "").strip() or not row.get("must_include", "").strip() for row in TESTS):
    raise ValueError("Every test row must include a question and must_include value")

def run_tests(active_chain, tests=TESTS):
    results = []
    for row in tests:
        answer = active_chain.invoke({"history": [], "question": row["question"]})
        required = row["must_include"].strip()
        passed = required.casefold() in answer.casefold()
        results.append({"passed": passed, "answer": answer})
        print("PASS" if passed else "FAIL", "|", row["question"], "| expected:", required, "|", answer[:100])

    passed_count = sum(result["passed"] for result in results)
    pass_rate = passed_count / len(results)
    print(f"\n{passed_count}/{len(results)} passed ({pass_rate:.0%})")
    return {"passed": passed_count, "total": len(results), "pass_rate": pass_rate, "answers": [r["answer"] for r in results]}


In [ ]:
baseline_results = run_tests(chain)


## Improve the prompt

One change: require the exact owner-check phrase when the answer is not in the bakery facts. Compare the pass rate with the baseline.


In [ ]:
old_instruction = 'say "I\'ll check with the owner."'
new_instruction = 'reply with the exact phrase "I\'ll check with the owner."'
SYSTEM_IMPROVED = SYSTEM.replace(old_instruction, new_instruction, 1)
if SYSTEM_IMPROVED == SYSTEM:
    raise ValueError("Could not apply the prompt improvement")

improved_chain = make_chain(SYSTEM_IMPROVED, TEMPERATURE)
improved_results = run_tests(improved_chain)
delta = improved_results["pass_rate"] - baseline_results["pass_rate"]
print(f"Baseline: {baseline_results['pass_rate']:.0%}; improved: {improved_results['pass_rate']:.0%}")
if delta > 0:
    print("Pass rate went up.")
elif delta < 0:
    print("Pass rate went down.")
else:
    print("Pass rate stayed the same.")


## Temperature experiment

Use temperature 1.0 and run the same tests twice. The cells compare pass rates and count responses that changed.


In [ ]:
temperature_chain = make_chain(SYSTEM_IMPROVED, temperature=1.0)

print("Temperature 1.0 ? run 1")
temperature_run_1 = run_tests(temperature_chain)
print("\nTemperature 1.0 ? run 2")
temperature_run_2 = run_tests(temperature_chain)

changed_answers = sum(
    first != second
    for first, second in zip(temperature_run_1["answers"], temperature_run_2["answers"])
)
print(f"\nPass rates: {temperature_run_1['pass_rate']:.0%} and {temperature_run_2['pass_rate']:.0%}")
print(f"Responses that changed: {changed_answers}/{len(TESTS)}")


## Agent or not?

- Calculate an order total from item prices and quantities ? **plain code**.
- Answer one customer question about prices, eggless options, or notice ? **one LLM call**.
- Take a custom cake order, ask for missing details, check availability, and submit it to an order system ? **agent** (it needs tools for availability and order submission).


In [ ]:
from langchain.agents import create_agent

agent_order = create_agent(model=llm, tools=[], system_prompt=SYSTEM)
result_order = agent_order.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result_order["messages"][-1].content)
